# 10. PyTorch Tutorial 10 - Dataset Transforms

torchvision.datasets contains various image datasets that are commonly used for computer vision and machine learning. Some of the key datasets in torchvision.datasets are:

- MNIST: Contains 70,000 grayscale images of handwritten digits from 0 to 9, with 60,000 training images and 10,000 testing images.

- Fashion-MNIST: Similar to MNIST but contains fashion products instead of handwritten digits. Contains 60,000 training images and 10,000 testing images of clothes.   

- CIFAR: Contains various small object datasets. CIFAR-10 contains 60,000 32x32 color images across 10 classes. CIFAR-100 contains 100 classes with 600 images each.

- ImageFolder: Can be used to load images from a directory structure where each subdirectory contains images of one class. Useful for loading custom datasets.

- SBDataset: Similar to ImageFolder but used for stereo image datasets that have left and right images for each sample.

Transforms in torchvision are operations that can be applied to the image datasets to augment the data or modify the images. Some common transforms are:

- Resize: Resizes images to a specific size.
- CenterCrop: Crops images to a centered square of a specific size. 
- RandomCrop: Crops a random square region of a specific size from images.
- RandomHorizontalFlip: Randomly flips images horizontally. 
- RandomRotation: Randomly rotates images by a specific degree.
- Convert to Tensor: Converts PIL Image or numpy ndarray images to torch.Tensor.
- Normalize: Normalizes image pixels by mean and standard deviation.

So in summary, torchvision.datasets contains common image datasets and torchvision.transforms contains  operations that can be applied to those datasets to modify or augment the images for training deep learning models.

In [2]:
import torch
import torchvision
from torch.utils.data import Dataset
import numpy as np

In [3]:
class WineDataset(Dataset):

    def __init__(self, transform=None):
        xy = np.loadtxt('./pytorchTutorial-master/data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = xy.shape[0]

        # note that we do not convert to tensor here
        self.x_data = xy[:, 1:]
        self.y_data = xy[:, [0]]

        self.transform = transform

    def __getitem__(self, index):
        sample = self.x_data[index], self.y_data[index]

        if self.transform:
            sample = self.transform(sample)

        return sample

    def __len__(self):
        return self.n_samples

### Custom Transforms

In [4]:
# implement __call__(self, sample)
class ToTensor:
    # Convert ndarrays to Tensors
    def __call__(self, sample):
        inputs, targets = sample
        return torch.from_numpy(inputs), torch.from_numpy(targets)

class MulTransform:
    # multiply inputs with a given factor
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, sample):
        inputs, targets = sample
        inputs *= self.factor
        return inputs, targets

In [6]:
print('Without Transform')
dataset = WineDataset()
first_data = dataset[0]
features, labels = first_data
print(type(features), type(labels))
print(features, labels)

Without Transform
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
 2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03] [1.]


### Single transform

In [7]:
print('\nWith Tensor Transform')
dataset = WineDataset(transform=ToTensor())
first_data = dataset[0]
features, labels = first_data
print(type(features), type(labels))
print(features, labels)


With Tensor Transform
<class 'torch.Tensor'> <class 'torch.Tensor'>
tensor([1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
        3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
        1.0650e+03]) tensor([1.])


### Multiple transforms : torchvision.transforms.Compose([Transf1(), Transf2() ])

In [8]:
print('\nWith Tensor and Multiplication Transform')
composed = torchvision.transforms.Compose([ToTensor(), MulTransform(4)])
dataset = WineDataset(transform=composed)
first_data = dataset[0]
features, labels = first_data
print(type(features), type(labels))
print(features, labels)


With Tensor and Multiplication Transform
<class 'torch.Tensor'> <class 'torch.Tensor'>
tensor([5.6920e+01, 6.8400e+00, 9.7200e+00, 6.2400e+01, 5.0800e+02, 1.1200e+01,
        1.2240e+01, 1.1200e+00, 9.1600e+00, 2.2560e+01, 4.1600e+00, 1.5680e+01,
        4.2600e+03]) tensor([1.])
